# Annual Precipitation Percentile Heatmap Analysis

## Input Data
- **Zarr stores** with 15-minute precipitation statistics by percentile band
  - Distribution data (PRECIP_PERIOD): Count of 15-min periods in each percentile band
  - Accumulation data (PRECIP_SUM): Total precipitation accumulation per band
- **Epochs**: HIST (1990-2005), MID (2050-2065), EOC (2085-2100)
- **Scenarios**: RCP4.5 and RCP8.5
- **Spatial**: 7 NCA regions with regional masks
- **Format**: Compressed Zarr datasets

## Processing Steps
1. **Load masked regional data**: Apply NCA region masks to precipitation datasets
2. **Calculate percentile band statistics**:
   - Temporal sum for each grid cell (across all time periods)
   - Spatial average across region (equal weight per grid cell)
3. **Compute percent changes**: Future epochs vs. historical baseline
4. **Generate visualizations**:
   - Small multiples bar charts (region × scenario × metric)
   - 2×2 heatmap grid 

## Output
- **Bar chart figures**: Individual region files + combined master figure
- **Heatmap figure**: 2×2 grid (RCP4.5/8.5 × Occurrences/Accumulation, within each MID and EOC)

In [2]:
# ============================================================================
# CELL 1: Imports and Configuration
# ============================================================================

import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as path_effects
import pandas as pd
import xarray as xr

# Force non-interactive backend to prevent display issues
matplotlib.use('Agg')
matplotlib.rcdefaults()
plt.close('all')

# ============================================================================
# Regional and Scenario Configuration
# ============================================================================

# NCA regions to process
regions = [
    "Midwest",
    "Northeast",
    "NorthernGreatPlains",
    "Northwest",
    "Southeast",
    "SouthernGreatPlains",
    "Southwest"
]

# RCP scenarios with file suffixes for mid-century and end-of-century
scenarios = [
    {"name": "4.5", "mid_suffix": "mid4p5", "eoc_suffix": "eoc4p5"},
    {"name": "8.5", "mid_suffix": "mid8p5", "eoc_suffix": "eoc8p5"}
]

# ============================================================================
# Path Configuration (relative paths)
# ============================================================================

base_path = "./combined_datasets"  # Zarr datasets for precipitation statistics
path_mask = "./masking"  # Regional mask files
output_dir = "./figures"  # Output directory for figures

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

print("Configuration complete.")
print(f"  Regions: {len(regions)}")
print(f"  Scenarios: {[s['name'] for s in scenarios]}")
print(f"  Data path: {base_path}")
print(f"  Mask path: {path_mask}")
print(f"  Output path: {output_dir}")

Configuration complete.
  Regions: 7
  Scenarios: ['4.5', '8.5']
  Data path: ./combined_datasets
  Mask path: ./masking
  Output path: ./figures


## Helper Functions

Core processing functions for percentile band analysis:
- `create_readable_labels()`: Convert numeric band codes to human-readable labels
- `process_percentile_bands()`: Calculate spatial averages with equal grid cell weighting
- `safe_percent_change()`: Compute percent change with division-by-zero protection
- `load_and_process_data()`: Full pipeline for loading, masking, and processing regional data

In [3]:
# ============================================================================
# CELL 2: Helper Functions
# ============================================================================

def create_readable_labels(labels):
    """
    Create human-readable labels for the percentile bands.
    
    Args:
        labels: List of percentile band codes (e.g., '0-0', '9-99')
    
    Returns:
        List of readable labels (e.g., 'None-Trace', '90th-99th')
    """
    readable_labels = []
    for label in labels:
        if label == '0-0':
            readable_labels.append('None-Trace')
        else:
            parts = label.split('-')
            if len(parts) == 2:
                # Map numeric codes to percentile labels
                label_map = {
                    ('0', '1'): 'Trace-10th',
                    ('1', '2'): '10th-20th',
                    ('2', '3'): '20th-30th',
                    ('3', '4'): '30th-40th',
                    ('4', '5'): '40th-50th',
                    ('5', '6'): '50th-60th',
                    ('6', '7'): '60th-70th',
                    ('7', '8'): '70th-80th',
                    ('8', '9'): '80th-90th',
                    ('9', '99'): '90th-99th',
                    ('99', '0'): '99th+' #yes, this format is weird... it is just a label
                }
                readable_labels.append(label_map.get(tuple(parts), f'{parts[0]}th-{parts[1]}th'))
    return readable_labels


def process_percentile_bands(dist_ds, accum_ds):
    """
    Process percentile bands using statistically sound spatial averaging:
    1. Calculate temporal sum for each grid cell
    2. Compute spatial average across all grid cells (equal weighting)
    
    Args:
        dist_ds: Distribution dataset (PRECIP_PERIOD counts)
        accum_ds: Accumulation dataset (PRECIP_SUM totals)
    
    Returns:
        Tuple of (percentile_labels, distribution_means, accumulation_means)
    """
    # Get unique percentile labels and sort (with '0-0' first)
    percentile_labels = sorted(
        list(set(dist_ds.percentile_label.values)), 
        key=lambda x: float(x.split('-')[0]) if x != '0-0' else -1
    )
    
    if '0-0' in percentile_labels:
        percentile_labels.remove('0-0')
        percentile_labels = ['0-0'] + percentile_labels
    
    # Calculate means for each percentile band
    dist_means = {}
    accum_means = {}
    
    for label in percentile_labels:
        # Select data for current percentile band
        dist_band = dist_ds.isel(
            concat_dim=[i for i, plabel in enumerate(dist_ds.percentile_label.values) 
                       if plabel == label]
        )
        accum_band = accum_ds.isel(
            concat_dim=[i for i, plabel in enumerate(accum_ds.percentile_label.values) 
                        if plabel == label]
        )
        
        # Temporal sum over concat_dim, then spatial mean
        dist_sum = dist_band.PRECIP_PERIOD.sum(dim='concat_dim')
        accum_sum = accum_band.PRECIP_SUM.sum(dim='concat_dim')
        
        # Spatial mean (excluding NaN from mask)
        dist_means[label] = float(dist_sum.mean(skipna=True).values)
        accum_means[label] = float(accum_sum.mean(skipna=True).values)
    
    return percentile_labels, dist_means, accum_means


def safe_percent_change(new_val, old_val, max_val=1000):
    """
    Calculate percent change with safety checks.
    
    Args:
        new_val: Future value
        old_val: Historical baseline value
        max_val: Maximum absolute percent change to return (clipping threshold)
    
    Returns:
        Percent change, or 0 if baseline is zero/NaN
    """
    if old_val == 0 or np.isnan(old_val) or np.isnan(new_val):
        return 0
    
    percent_change = ((new_val / old_val) - 1) * 100
    
    if np.isinf(percent_change) or np.isnan(percent_change):
        return 0
    
    return np.clip(percent_change, -max_val, max_val)


def load_and_process_data(region, scenario):
    """
    Load and process data for a specific region and scenario.
    
    Pipeline:
    1. Load historical, mid-century, and end-of-century datasets
    2. Apply regional mask
    3. Process percentile bands (temporal sum → spatial average)
    4. Calculate percent changes vs. historical baseline
    
    Args:
        region: Region name (e.g., 'Midwest')
        scenario: Scenario dict with name and file suffixes
    
    Returns:
        Dict with readable labels and DataFrames of percent changes
    """
    scenario_name = scenario["name"]
    mid_suffix = scenario["mid_suffix"]
    eoc_suffix = scenario["eoc_suffix"]
    
    # Define paths for distribution and accumulation datasets
    hist_dist_path = os.path.join(base_path, "dist_hist_dataset.zarr")
    mid_dist_path = os.path.join(base_path, f"dist_{mid_suffix}_dataset.zarr")
    eoc_dist_path = os.path.join(base_path, f"dist_{eoc_suffix}_dataset.zarr")

    hist_accum_path = os.path.join(base_path, "accum_hist_dataset.zarr")
    mid_accum_path = os.path.join(base_path, f"accum_{mid_suffix}_dataset.zarr")
    eoc_accum_path = os.path.join(base_path, f"accum_{eoc_suffix}_dataset.zarr")
    
    print(f"  Loading mask for {region}...")
    # Load regional mask
    mask = xr.open_dataset(os.path.join(path_mask, f"Masked_NCA_{region}.nc"))
    
    print(f"  Loading datasets for RCP{scenario_name}...")
    # Load all epoch datasets
    hist_dist = xr.open_dataset(hist_dist_path, engine="zarr")
    mid_dist = xr.open_dataset(mid_dist_path, engine="zarr")
    eoc_dist = xr.open_dataset(eoc_dist_path, engine="zarr")

    hist_accum = xr.open_dataset(hist_accum_path, engine="zarr")
    mid_accum = xr.open_dataset(mid_accum_path, engine="zarr")
    eoc_accum = xr.open_dataset(eoc_accum_path, engine="zarr")
    
    print(f"  Applying {region} mask...")
    # Apply mask to all datasets
    hist_dist = hist_dist.where(mask["mask"])
    mid_dist = mid_dist.where(mask["mask"])
    eoc_dist = eoc_dist.where(mask["mask"])

    hist_accum = hist_accum.where(mask["mask"])
    mid_accum = mid_accum.where(mask["mask"])
    eoc_accum = eoc_accum.where(mask["mask"])
    
    print(f"  Processing percentile bands...")
    # Process each epoch
    hist_labels, hist_dist_totals, hist_accum_totals = process_percentile_bands(hist_dist, hist_accum)
    mid_labels, mid_dist_totals, mid_accum_totals = process_percentile_bands(mid_dist, mid_accum)
    eoc_labels, eoc_dist_totals, eoc_accum_totals = process_percentile_bands(eoc_dist, eoc_accum)
    
    # Ensure common percentile bands across all epochs
    common_labels = sorted(
        list(set(hist_labels) & set(mid_labels) & set(eoc_labels)), 
        key=lambda x: float(x.split('-')[0]) if x != '0-0' else -1
    )

    if '0-0' in common_labels:
        common_labels.remove('0-0')
        common_labels = ['0-0'] + common_labels
    
    # Create readable labels
    readable_labels = create_readable_labels(common_labels)
    
    # Calculate percent changes with safety checks
    percent_change_mid_dist = pd.DataFrame({
        'Percentile Band': common_labels,
        'Readable Label': readable_labels,
        'Percent Change': [safe_percent_change(mid_dist_totals[label], hist_dist_totals[label]) 
                          for label in common_labels]
    })
    
    percent_change_eoc_dist = pd.DataFrame({
        'Percentile Band': common_labels,
        'Readable Label': readable_labels,
        'Percent Change': [safe_percent_change(eoc_dist_totals[label], hist_dist_totals[label]) 
                          for label in common_labels]
    })
    
    percent_change_mid_accum = pd.DataFrame({
        'Percentile Band': common_labels,
        'Readable Label': readable_labels,
        'Percent Change': [safe_percent_change(mid_accum_totals[label], hist_accum_totals[label]) 
                          for label in common_labels]
    })
    
    percent_change_eoc_accum = pd.DataFrame({
        'Percentile Band': common_labels,
        'Readable Label': readable_labels,
        'Percent Change': [safe_percent_change(eoc_accum_totals[label], hist_accum_totals[label]) 
                          for label in common_labels]
    })
    
    return {
        'readable_labels': readable_labels,
        'mid_dist': percent_change_mid_dist,
        'eoc_dist': percent_change_eoc_dist,
        'mid_accum': percent_change_mid_accum,
        'eoc_accum': percent_change_eoc_accum
    }

In [ ]:
# ============================================================================
# CELL 3: Data Collection
# ============================================================================

print("Collecting data using improved spatial averaging methodology...")
all_data = {}

for region in regions:
    print(f"\nProcessing region: {region}")
    all_data[region] = {}
    
    for scenario in scenarios:
        print(f"  Processing scenario: RCP{scenario['name']}")
        all_data[region][scenario['name']] = load_and_process_data(region, scenario)

print("\nData collection complete!")

## Small Multiples Visualization

Generate individual bar chart figures for each region showing:
- 8 subplots per region (2 scenarios × 4 metrics)
- Fixed y-axis limits (-40% to 90%) for cross-region comparison
- Grid lines at 10% increments with emphasized zero line
- Color coding: dark gray (increases), light gray (decreases)

In [ ]:
# ============================================================================
# CELL 4: Create Small Multiples with One Row Per Figure
# ============================================================================

# Fixed axis limits for all plots (enables cross-region comparison)
YMIN = -40
YMAX = 90

print(f"\nCreating small multiples with FIXED y-axis limits: [{YMIN}, {YMAX}]")

# Create one figure per region
for r, region in enumerate(regions):
    fig = plt.figure(figsize=(16, 2))
    row_axes = []
    
    # Create 8 subplots (2 scenarios × 4 plot types)
    for j, scenario in enumerate(scenarios):
        base_col = j * 4
        data = all_data[region][scenario['name']]
        
        # Plot configuration
        plot_types = ['mid_dist', 'eoc_dist', 'mid_accum', 'eoc_accum']
        titles = [
            f"Mid RCP{scenario['name']} Occur.",
            f"EOC RCP{scenario['name']} Occur.",
            f"Mid RCP{scenario['name']} Accum.",
            f"EOC RCP{scenario['name']} Accum."
        ]
        
        for i, (plot_type, title) in enumerate(zip(plot_types, titles)):
            pos = base_col + i + 1
            ax = fig.add_subplot(1, 8, pos)
            row_axes.append(ax)
            
            # Set fixed limits BEFORE plotting
            ax.set_ylim(YMIN, YMAX)
            
            # Get data and plot bars
            plot_data = data[plot_type]
            changes = plot_data['Percent Change'].values
            colors = ['darkgray' if val >= 0 else 'lightgray' for val in changes]
            bars = ax.bar(range(len(changes)), changes, color=colors, 
                         edgecolor='black', linewidth=0.5)
            
            # Add grid lines
            for y in range(YMIN, YMAX+1, 10):
                if y != 0:
                    ax.axhline(y=y, color='lightgray', linestyle='-', 
                              linewidth=0.5, alpha=0.5, zorder=0)
            
            # Emphasize zero line
            ax.axhline(y=0, color='black', linestyle='-', 
                      linewidth=0.8, alpha=0.7, zorder=1)
            
            # Title for first region only
            if r == 0:
                ax.set_title(title, fontsize=8, pad=4)
            
            # Remove spines
            for spine in ax.spines.values():
                spine.set_visible(False)
            
            # X-axis labels (show 5 evenly spaced for readability)
            n_labels = len(plot_data['Readable Label'])
            if n_labels > 10:
                indices = np.linspace(0, n_labels-1, 5).astype(int)
                labels = [plot_data['Readable Label'].iloc[idx] for idx in indices]
                ax.set_xticks(indices)
                ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=6)
            else:
                ax.set_xticks(range(n_labels))
                ax.set_xticklabels(plot_data['Readable Label'], rotation=45, ha='right', fontsize=6)
            
            # Y-axis labels only for first subplot
            if i == 0:
                ax.set_yticks(range(YMIN, YMAX+1, 10))
                ax.tick_params(axis='y', labelsize=6)
            else:
                ax.set_yticks([])
            
            # Region label on left side
            if i == 0:
                ax.text(-0.5, 0.5, region, transform=ax.transAxes, 
                       va='center', ha='right', fontsize=10, fontweight='bold',
                       rotation=90)
    
    # Verify consistent axis limits
    for ax in row_axes:
        assert ax.get_ylim() == (YMIN, YMAX), f"Axis y-limits are wrong: {ax.get_ylim()}"
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.25)
    
    # Save individual region figure
    output_path = os.path.join(output_dir, f"{region}_precipitation_changes.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

print("Small multiples created successfully!")

In [ ]:
# ============================================================================
# CELL 5: Combine All Region Figures into One Master Figure
# ============================================================================

print("\nCombining all region figures into master figure...")

# Create master figure
master_fig = plt.figure(figsize=(12, 24))

# Add titles and labels
master_fig.suptitle('Percent Change in Precipitation by Percentile Band Across Regions and Scenarios', 
                   fontsize=14, y=0.98)
master_fig.text(0.02, 0.5, 'Percent Change (%)', va='center', rotation=90, fontsize=12)
master_fig.text(0.5, 0.02, 'Percentile Bands (from None-Trace to 99th+)', ha='center', fontsize=12)

# Load and place each region's figure
for r, region in enumerate(regions):
    region_path = os.path.join(output_dir, f"{region}_precipitation_changes.png")
    region_fig = plt.imread(region_path)
    
    ax = master_fig.add_subplot(len(regions), 1, r+1)
    ax.imshow(region_fig)
    ax.axis('off')

# Adjust layout and save
plt.tight_layout(rect=[0.03, 0.03, 0.97, 0.95])

output_path = os.path.join(output_dir, "precipitation_percent_change_small_multiples.png")
plt.savefig(output_path, dpi=300)

output_path = os.path.join(output_dir, "precipitation_percent_change_small_multiples.pdf")
plt.savefig(output_path)

print("Master figure saved successfully!")
print(f"All plots use identical y-axis limits: [{YMIN}, {YMAX}]")

## Heatmap Visualization

Generate 2×2 heatmap grid with:
- **Layout**: RCP4.5/8.5 (rows) × Occurrences/Accumulation (columns)
- **Color scale**: Diverging BrBG colormap (-50% to +50%)
- **Annotations**: Percent change values with white outline for readability
- **Region labels**: Abbreviated (e.g., MW, NE) with Mid/EOC distinction
- **Output**: PNG (300 dpi) and PDF formats

In [ ]:
# ============================================================================
# CELL 6: Heatmap Visualization with Diverging Colormap - 2×2 Layout
# ============================================================================

print("Creating heatmap visualization with improved text annotations...")

# Get percentile labels from any region/scenario
percentile_labels = all_data[regions[0]][scenarios[0]['name']]['mid_dist']['Readable Label'].values

# Set fixed color scale limits
vmin = -50  
vmax = 50

# Create 2×2 figure
fig, axes = plt.subplots(2, 2, figsize=(9.75, 10.5), sharex='col', sharey='row')
axes_flat = axes.flatten()

# Use BrBG colormap for good grayscale rendering
cmap = plt.cm.BrBG

# Define plot configuration
plot_configs = [
    {'scenario': '4.5', 'data_type': 'dist', 'title': 'RCP4.5 Occurrences', 'ax_idx': 0},
    {'scenario': '4.5', 'data_type': 'accum', 'title': 'RCP4.5 Accumulation', 'ax_idx': 1},
    {'scenario': '8.5', 'data_type': 'dist', 'title': 'RCP8.5 Occurrences', 'ax_idx': 2},
    {'scenario': '8.5', 'data_type': 'accum', 'title': 'RCP8.5 Accumulation', 'ax_idx': 3}
]

# Region abbreviations
region_abbrevs = {
    "Midwest": "MW",
    "Northeast": "NE", 
    "NorthernGreatPlains": "NGP",
    "Northwest": "NW",
    "Southeast": "SE",
    "SouthernGreatPlains": "SGP",
    "Southwest": "SW"
}

# Create each heatmap
for config in plot_configs:
    ax = axes_flat[config['ax_idx']]
    
    # Prepare data array (rows: regions × 2 epochs, columns: percentile bands)
    data = np.zeros((len(regions) * 2, len(percentile_labels)))
    row_labels = []
    
    for r_idx, region in enumerate(regions):
        mid_row = r_idx * 2
        eoc_row = r_idx * 2 + 1
        
        abbrev = region_abbrevs.get(region, region[:2])
        row_labels.append(f"{abbrev} Mid")
        row_labels.append(f"{abbrev} EOC")
        
        # Get data keys
        mid_key = f"mid_{config['data_type']}"
        eoc_key = f"eoc_{config['data_type']}"
        
        if config['scenario'] in all_data[region]:
            data[mid_row, :] = all_data[region][config['scenario']][mid_key]['Percent Change'].values
            data[eoc_row, :] = all_data[region][config['scenario']][eoc_key]['Percent Change'].values
    
    # Create heatmap
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
    
    # Add text annotations with white outline for readability
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data[i, j]
            text = f"{value:.0f}" if abs(value) >= 10 else f"{value:.1f}"
            
            ax.text(j, i, text, ha="center", va="center", 
                    color='black', fontsize=7, fontweight='bold',
                    path_effects=[
                        path_effects.withStroke(linewidth=2, foreground='white')
                    ])
    
    # Set title
    ax.set_title(config['title'], fontsize=12)
    
    # Y-axis labels (left column only)
    if config['ax_idx'] % 2 == 0:
        ax.set_yticks(np.arange(len(row_labels)))
        ax.set_yticklabels(row_labels, fontsize=9)
    
    # X-axis labels (bottom row only)
    if config['ax_idx'] >= 2:
        ax.set_xticks(np.arange(len(percentile_labels)))
        ax.set_xticklabels(percentile_labels, rotation=45, ha='right', fontsize=9)
    
    # Add lines to separate region pairs
    for y in np.arange(1.5, len(row_labels), 2):
        ax.axhline(y=y, color='black', linestyle='-', linewidth=1.0, alpha=0.7)

# Add colorbar
cax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
cbar = plt.colorbar(im, cax=cax, extend='both')
cbar.set_label('Δ (%)', fontsize=12)

# Save figure
plt.tight_layout(rect=[0, 0, 0.92, 0.95])

output_path = os.path.join(output_dir, 'precipitation_heatmap_annotated.png')
plt.savefig(output_path, dpi=300)

output_path = os.path.join(output_dir, 'precipitation_heatmap_annotated.pdf')
plt.savefig(output_path)

print("Annotated heatmap visualization complete!")